# 🚀 LAB GUIDE — PRODUCTION-GRADE GRAPHRAG VS FLAT RAG

**Thời lượng:** 120 phút  
**Môi trường:** Windows + `.venv` + JupyterLab + Neo4j Docker; cũng hỗ trợ Colab khi có runtime files  
**Dữ liệu:** HackerNoon Tech Company News Data Dump (bản thu gọn do giảng viên cung cấp)  
**Công cụ:** Học viên được dùng AI Coding Agent, nhưng phải tự thiết kế, kiểm thử và giải thích logic.

## 🎯 Mục tiêu
1. Xây dựng Hybrid GraphRAG end-to-end.
2. Xử lý Coreference Resolution, Entity Resolution và Super-node Mitigation.
3. Bulk insert bằng `UNWIND`, không insert từng row.
4. So sánh Flat RAG và GraphRAG bằng Golden Dataset + LLM-as-a-Judge.
5. Đo quality, latency và token usage.
6. Giải thích kiến trúc và failure modes.

> Notebook là **reference lab guide**: có code khung chạy được nhưng vẫn yêu cầu học viên thay prompt/threshold/retrieval policy và thuyết minh lựa chọn.

**Bản triển khai local:** [WORKFLOW.md](WORKFLOW.md), [lab19_runtime.py](lab19_runtime.py), [tests/test_base.py](tests/test_base.py).
Cell in PENDING khi thiếu dữ liệu/secrets; trạng thái này không có nghĩa pipeline đã hoàn thành.


## ⏳ Timeline

| Phút | Nội dung |
|---|---|
| 00–15 | Setup, load, dedup, chunk, coreference |
| 15–45 | NER/RE, entity resolution, Neo4j bulk insert |
| 45–75 | Flat RAG, graph traversal, hybrid retrieval |
| 75–105 | Golden Dataset, LLM-as-a-Judge, comparison |
| 105–120 | Failure-mode tests, bonus, export, thuyết minh |

### Scale guard
Trong lab 2 giờ, không nên gửi toàn bộ 350MB qua LLM. Mặc định dùng subset:
- `LAB_MAX_ARTICLES = 1500`
- `LAB_MAX_CHUNKS = 3000`
- `EXTRACTION_MAX_CHUNKS = 400`

Kiến trúc phải scale được; volume trong giờ lab chỉ dùng để chứng minh pipeline.

# PHẦN 1 — SETUP & PREPROCESSING

### Local Windows
Secrets được nạp từ `.env` bằng `python-dotenv`; không in hoặc commit secrets.
Neo4j local dùng `bolt://localhost:7687`. Dataset, checkpoint và exports dùng đường dẫn trong repo.
Cấu hình Groq cho extraction/generation; Judge dùng OpenAI, Groq hoặc Gemini theo `JUDGE_PROVIDER`.
Mặc định dùng `JUDGE_PROVIDER=groq` và model Groq phù hợp; pipeline/Judge dùng chung `GROQ_API_KEY`.
Trong Colab, cần có các file runtime/support và có thể dùng Colab Secrets.


In [1]:
# 1.1 — Local environment: install only when recreating .venv
import sys, importlib.util
required_modules = ['neo4j', 'pandas', 'numpy', 'sentence_transformers', 'faiss', 'groq', 'openai', 'dotenv']
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
print('Kernel:', sys.executable)
print('Missing modules:', missing_modules)
if missing_modules:
    raise RuntimeError('Install requirements.txt plus jupyterlab/ipykernel in this kernel environment.')

Kernel: D:\VinUniversity\K4-DAY19-VuDuyDiep-2A202602703\.venv\Scripts\python.exe
Missing modules: []


In [2]:
# 1.2 — Imports, .env and Windows paths
import importlib
import pandas as pd
from IPython.display import display
import lab19_runtime as lab
import lab19_support as support
lab = importlib.reload(lab)  # Re-read .env after editing keys; no secrets are displayed.
print('Repo:', lab.ROOT)
print('Scale guard:', lab.LAB_MAX_ARTICLES, lab.LAB_MAX_CHUNKS, lab.EXTRACTION_MAX_CHUNKS)
print('Runtime implementation: lab19_runtime.py; workflow: WORKFLOW.md')
data_ready = graph_ready = retrieval_ready = False
eval_results_df = None

D:\VinUniversity\K4-DAY19-VuDuyDiep-2A202602703\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo: D:\VinUniversity\K4-DAY19-VuDuyDiep-2A202602703
Scale guard: 1500 3000 400
Runtime implementation: lab19_runtime.py; workflow: WORKFLOW.md


## 1.3 — Corpus nguồn và streaming có giới hạn

Ưu tiên bản `hackernoon_subset.csv` gốc để đối chiếu Golden. Script `scripts/download_corpus.py`
stream tối đa 5.000 dòng/300 MiB từ Hugging Face; dataset có gated access nên cần `HF_TOKEN` hợp lệ.
Tải lại không bảo đảm thứ tự row giống bản Golden: pipeline khớp URL hoặc title/date trước khi sample.
Giữ scale guard 1.500 articles/3.000 chunks/400 extraction chunks cho phần xử lý.


In [3]:
# 1.3 — Use an existing corpus; gated download requires a configured HF_TOKEN
if lab.DATA_PATH.exists():
    print('Using corpus:', lab.DATA_PATH)
elif lab.HF_TOKEN:
    import subprocess
    subprocess.run([sys.executable, str(lab.ROOT / 'scripts/download_corpus.py')], check=True)
else:
    print('PENDING W1: add the original corpus at data/hackernoon_subset.csv, or configure HF_TOKEN.')

Using corpus: D:\VinUniversity\K4-DAY19-VuDuyDiep-2A202602703\data\hackernoon_subset.csv


In [4]:
# 1.4 — Driver, schema, secret status and minimal provider probes
preflight_df = lab.preflight(check_llm=True)
display(preflight_df)
neo4j_ready = preflight_df.loc[preflight_df.check.eq('neo4j_driver'), 'status'].eq('passed').all()
llm_ready = not preflight_df.status.isin(['missing_or_placeholder', 'failed']).any()
print('Neo4j ready:', neo4j_ready, '| LLM ready:', llm_ready)

Neo4j connected.
Schema ready.


,check,status,detail
0,JUDGE_PROVIDER,configured,groq
1,NEO4J_URI,configured,
2,NEO4J_PASSWORD,configured,
3,GROQ_API_KEY,configured,
4,GROQ_MODEL,configured,
5,JUDGE_MODEL,configured,
6,corpus,present,
7,HF_TOKEN,configured,
8,neo4j_driver,passed,194 existing nodes; lab queries are namespaced
9,groq_request,passed,


Neo4j ready: True | LLM ready: True


In [5]:
# 1.5 — Dedup, source identity, coverage-selected sample and chunks
if lab.DATA_PATH.exists():
    try:
        news_df, chunks_df, extraction_source, golden_df, golden_coverage_df = support.prepare_corpus(lab)
        data_ready = True
        display(chunks_df.head())
        display(golden_coverage_df)
        print('Selected articles/chunks/extraction:', len(news_df), len(chunks_df), len(extraction_source))
    except Exception as error:
        print('PENDING W1/W2:', lab.safe_error(error))
else:
    print('PENDING W1/W2: corpus is missing; no synthetic corpus is substituted.')

Exact dedup: 2,695 -> 2,119


Chunking:   0%|          | 0/1500 [00:00<?, ?it/s]

Chunking:  90%|████████▉ | 1348/1500 [00:00<00:00, 13475.07it/s]

Chunking: 100%|██████████| 1500/1500 [00:00<00:00, 13470.28it/s]

,chunk_id,article_id,source_row_ids_json,title,published_date,text
0,613de2c24d2456048c08::c0000,613de2c24d2456048c08,[2532],Exclusive: Amazon has drawn thousands to try its AI service competing with Microsoft Google,2023-07-26,Exclusive: Amazon has drawn thousands to try its AI service competing with Microsoft Google. The company also announ...
1,ae3e878731488df141c0::c0000,ae3e878731488df141c0,"[2537, 3737]",Amazon has drawn thousands to try its AI service competing with Microsoft Google,2023-07-27,Amazon has drawn thousands to try its AI service competing with Microsoft Google. The company also announced new AI ...
2,0d53dc66fb96e3cb91e2::c0000,0d53dc66fb96e3cb91e2,[2905],Exclusive: Amazon's cloud unit is considering AMD's new AI chips,2023-06-14,Exclusive: Amazon's cloud unit is considering AMD's new AI chips. Amazon Web Services the world''s largest cloud com...
3,8d2f8b148829f4949b97::c0000,8d2f8b148829f4949b97,[3330],More tech companies take White House AI safety pledge,2023-09-12,More tech companies take White House AI safety pledge. IBM Adobe Salesforce and five more tech companies are making ...
4,486309b2b36023d69765::c0000,486309b2b36023d69765,[3357],3 Best Cloud Stocks to Buy in June,2023-06-01,3 Best Cloud Stocks to Buy in June. Microsoft''s market share in cloud computing puts it second with the potential t...


,id,group,corpus_covered,vector_covered,extraction_covered
0,G5000-26,multi-hop,True,True,True
1,G5000-27,cross-doc,True,True,True
2,G5000-28,multi-hop,True,True,True
3,G5000-29,cross-doc,True,True,True
4,G5000-41,factoid,True,True,True


Selected articles/chunks/extraction: 1500 1500 200


### 🎯 AI Coding Agent Challenge A — Near Dedup
Exact hash không bắt được bài repost/near-duplicate.

Hãy dùng AI Agent thiết kế thêm **MinHash/LSH, SimHash hoặc embedding+ANN**.  
**Không chấp nhận** pairwise cosine `O(N²)` trên toàn dataset.

Trong báo cáo nêu:
1. threshold,
2. false positive,
3. cách audit cặp bị merge.

In [6]:
# 1.6 — Retry/JSON implementation and provider configuration
from lab19_runtime import groq_chat, groq_json, parse_json_object
print('Generator model:', lab.GROQ_MODEL, '| Judge:', lab.JUDGE_PROVIDER, lab.JUDGE_MODEL)
print('Credentials are read from .env/Colab Secrets and never printed.')

Generator model: openai/gpt-oss-120b | Judge: groq openai/gpt-oss-120b
Credentials are read from .env/Colab Secrets and never printed.


## 1.7 — Coreference Resolution

Yêu cầu:
- chỉ resolve đại từ khi antecedent rõ trong cùng chunk,
- không invent fact,
- giữ nguyên số/ngày/ticker/product,
- ambiguity → giữ nguyên và log `unresolved_mentions`.

**Failure mode quan trọng:** false coreference → false edge.

In [7]:
# 1.7 — Conservative coreference with per-batch checkpoints and numeric guard
if data_ready and llm_ready:
    coref_df = lab.run_coref(extraction_source)
    extraction_source = extraction_source.merge(coref_df, on='chunk_id', validate='one_to_one')
    display(coref_df.head())
    display(coref_df.coref_status.value_counts())
else:
    print('PENDING W2: coreference requires the corpus and configured LLM.')

Coref:   0%|          | 0/25 [00:00<?, ?it/s]

Coref:  72%|███████▏  | 18/25 [00:00<00:00, 179.24it/s]

Coref: 100%|██████████| 25/25 [00:00<00:00, 177.71it/s]

,chunk_id,resolved_text,unresolved_mentions,coref_status
0,613de2c24d2456048c08::c0000,Exclusive: Amazon has drawn thousands to try its AI service competing with Microsoft Google. The company also announ...,[],unchanged
1,ae3e878731488df141c0::c0000,Amazon has drawn thousands to try its AI service competing with Microsoft Google. The company also announced new AI ...,[],unchanged
2,0d53dc66fb96e3cb91e2::c0000,Exclusive: Amazon's cloud unit is considering AMD's new AI chips. Amazon Web Services the world''s largest cloud com...,[],unchanged
3,8d2f8b148829f4949b97::c0000,More tech companies take White House AI safety pledge. IBM Adobe Salesforce and five more tech companies are making ...,[],unchanged
4,486309b2b36023d69765::c0000,3 Best Cloud Stocks to Buy in June. Microsoft''s market share in cloud computing puts it second with the potential t...,[],unchanged


coref_status
unchanged    178
resolved      21
rejected       1
Name: count, dtype: int64

# PHẦN 2 — TRIPLE EXTRACTION & NEO4J BULK INSERT (15–45')

## Graph schema
**Nodes:** `Company`, `Person`, `Technology` + base label `Entity`.

**Relations:** `ACQUIRED`, `DEVELOPED`, `INVESTED_IN`, `FOUNDED`, `WORKED_AT`, `PARTNERED_WITH`, `USES`, `LEADS`.

**Mỗi edge bắt buộc:** `source_chunk_id`, `published_date`; khuyến nghị thêm `evidence`, `confidence`.

> Relation type phải qua allowlist trước khi ghép vào Cypher.

In [8]:
# 2.1 — Schema-checked triples; evidence must be copied from original text
if data_ready and llm_ready:
    raw_triples_df, extraction_errors_df = lab.run_extraction(extraction_source)
    display(raw_triples_df.head())
    display(extraction_errors_df.head())
    print('Valid triples:', len(raw_triples_df), '| errors/rejections:', len(extraction_errors_df))
else:
    raw_triples_df = pd.DataFrame()
    print('PENDING W3: NER/RE has not run.')

NER+RE:   0%|          | 0/25 [00:00<?, ?it/s]

NER+RE:  68%|██████▊   | 17/25 [00:00<00:00, 164.06it/s]

NER+RE: 100%|██████████| 25/25 [00:00<00:00, 169.88it/s]

,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence
0,Amazon,Company,USES,Cohere,Company,613de2c24d2456048c08::c0000,2023-07-26,technology access from the startup Cohere,1.0
1,Hewlett Packard Enterprise,Company,ACQUIRED,Axis Security,Company,5107f0cd19116339b25d::c0000,2023-03-02,entered into a definitive agreement to acquire Axis Security,1.0
2,Google Cloud,Company,USES,Llama 2,Technology,7ca725290aab95bd56a7::c0000,2023-08-29,announcing availability of Llama 2,1.0
3,Google Cloud,Company,USES,Falcon LLM,Technology,7ca725290aab95bd56a7::c0000,2023-08-29,Technology Innovative Institute''s Falcon LLM a popular open-source model,1.0
4,Google Cloud,Company,USES,Claude 2,Technology,7ca725290aab95bd56a7::c0000,2023-08-29,pre-announcing Claude 2 from Anthropic,1.0


,start,chunk_id,error
0,0,7ca725290aab95bd56a7::c0000,Rejected invalid evidence/confidence/date
1,24,5de01398a4298b732892::c0000,Rejected invalid evidence/confidence/date
2,24,5de01398a4298b732892::c0000,Rejected invalid evidence/confidence/date
3,24,a35c246dcfe101d64398::c0000,Rejected invalid evidence/confidence/date
4,48,3a76b31b81939716924c::c0000,Rejected invalid evidence/confidence/date


Valid triples: 52 | errors/rejections: 10


## 2.2 — Entity Resolution bằng Vector Similarity

Pipeline:
1. Manual aliases cho ticker/tên rất phổ biến.
2. Embedding ANN candidate.
3. Lexical guard để giảm false merge.
4. Xuất audit table.

### 🎯 AI Coding Agent Challenge B
Cải tiến guard cho:
- ticker,
- suffix `Inc./Corp./Ltd.`,
- product chứa company name,
- người trùng họ/tên gần giống.

In [9]:
# 2.2 — Alias → normalized HNSW candidates → lexical/type guard → Union-Find
if not raw_triples_df.empty:
    entity_map, entity_resolution_audit_df = lab.build_resolution_map(raw_triples_df, threshold=0.90)
    triples_df = lab.canonicalize_triples(raw_triples_df, entity_map)
    display(entity_resolution_audit_df.head(20))
    print('Audit rows:', len(entity_resolution_audit_df))
else:
    print('PENDING W3: no extracted triples to resolve.')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1350.95it/s]

,type,left,right,similarity,decision,reason,canonical_left,canonical_right
0,Company,Amazon,Amazon Web Services,0.646989,REJECT_THRESHOLD,similarity_below_threshold,Amazon,Amazon Web Services
1,Company,Amazon,Amazon Publisher Services,0.585162,REJECT_THRESHOLD,similarity_below_threshold,Amazon,Amazon Publisher Services
2,Company,Amazon,Salesforce,0.537856,REJECT_THRESHOLD,similarity_below_threshold,Amazon,Salesforce
3,Company,Amazon,Apple,0.485086,REJECT_THRESHOLD,similarity_below_threshold,Amazon,Apple
4,Company,Cohere,Linde,0.367736,REJECT_THRESHOLD,similarity_below_threshold,Cohere,Linde
5,Company,Cohere,Worldpay,0.351073,REJECT_THRESHOLD,similarity_below_threshold,Cohere,Worldpay
6,Company,Cohere,Odyssey,0.338944,REJECT_THRESHOLD,similarity_below_threshold,Cohere,Odyssey
7,Company,Cohere,Thales,0.336762,REJECT_THRESHOLD,similarity_below_threshold,Cohere,Thales
8,Company,Hewlett Packard Enterprise,Microsoft,0.488284,REJECT_THRESHOLD,similarity_below_threshold,Hewlett Packard Enterprise,Microsoft
9,Company,Hewlett Packard Enterprise,Airbus,0.405509,REJECT_THRESHOLD,similarity_below_threshold,Hewlett Packard Enterprise,Airbus


Audit rows: 282


In [10]:
# 2.3 — Bulk UNWIND; nodes and relationships are scoped to LAB_NAMESPACE
if not raw_triples_df.empty and neo4j_ready:
    nodes_df = lab.build_nodes(triples_df)
    lab.bulk_insert_nodes(nodes_df)
    lab.bulk_insert_edges(triples_df)
    graph_ready = True
    print('Ingested nodes/triples:', len(nodes_df), len(triples_df))
else:
    print('PENDING W3: ingestion has not run.')

Ingested nodes/triples: 97 52


In [11]:
# 2.4 — Includes NULL, empty-string provenance and a real top-degree table
if graph_ready:
    graph_counts, top_degree_df = lab.graph_checks()
else:
    print('PENDING W3: graph integrity has not been measured.')

{'nodes': 97, 'edges': 52, 'invalid_provenance_edges': 0}


,id,name,type,degree
0,18a2f0e496fb60f2b80e8775,Google Cloud,Company,3
1,fc94d4789fde1f6e802475fd,L&T Technology Services,Company,2
2,51e652cc3cb71d3060de600c,Thales,Company,2
3,2618ac416fd176fc8751f56d,Data Innovations,Company,2
4,da83c25f2271b3c0df0cad56,AGC Biologics,Company,2
5,5d63f575c2fcc0b8f6e2e3c8,Freshworks,Company,2
6,1b8f7284287734ef1f96795d,Sineng Electric,Company,1
7,2a53b5e8507255a685aa84d7,Aqara,Company,1
8,0fccf849f555ed7fc05dba5e,Qualcomm,Company,1
9,24a29c9a8a01d23c25f3bb61,Associated Press,Company,1


# PHẦN 3 — FLAT RAG & HYBRID GRAPHRAG (45–75')

## Flat RAG baseline
Dùng cùng embedding/generator để comparison tập trung vào retrieval architecture.

In [12]:
# 3.1 — Flat RAG index over the same corpus chunks
if data_ready:
    lab.build_flat_index(chunks_df)
    print('Flat baseline ready; top-k = 6.')
else:
    print('PENDING W4: Flat index requires corpus chunks.')

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Batches:   8%|▊         | 1/12 [00:27<05:00, 27.31s/it]

Batches:  17%|█▋        | 2/12 [00:34<02:36, 15.63s/it]

Batches:  25%|██▌       | 3/12 [00:40<01:39, 11.05s/it]

Batches:  33%|███▎      | 4/12 [00:47<01:17,  9.69s/it]

...

Batches:  83%|████████▎ | 10/12 [01:16<00:09,  4.99s/it]

Batches:  92%|█████████▏| 11/12 [01:20<00:04,  4.77s/it]

Batches: 100%|██████████| 12/12 [01:23<00:00,  4.18s/it]

Batches: 100%|██████████| 12/12 [01:23<00:00,  6.92s/it]

Flat vectors: 1500
Flat baseline ready; top-k = 6.


## Graph retrieval flow
1. LLM trích seed entities.
2. Match seed trong Neo4j; fuzzy fallback bằng embedding.
3. BFS tối đa `max_hops`.
4. Nếu node degree > 100 → chỉ lấy tối đa 50 edge mới nhất.
5. Global edge cap để tránh context explosion.
6. Textualize subgraph có provenance.

In [13]:
# 3.2 — Seed resolution: exact aliases then embedding fallback 0.66
if graph_ready:
    lab.build_entity_matcher(nodes_df)
    retrieval_ready = True
else:
    print('PENDING W4: entity matcher requires canonical graph nodes.')

In [14]:
# 3.3 — Policy is tested offline separately from real-data graph measurements
print({'hops': 2, 'degree_threshold': lab.SUPER_NODE_DEGREE,
       'supernode_edge_cap': lab.SUPER_NODE_EDGE_CAP, 'global_edge_cap': lab.GLOBAL_EDGE_CAP,
       'graph_context_chars': lab.MAX_GRAPH_CONTEXT_CHARS})
print('Behavioral regression tests: python -m pytest tests/test_base.py -q')

{'hops': 2, 'degree_threshold': 100, 'supernode_edge_cap': 50, 'global_edge_cap': 250, 'graph_context_chars': 14000}
Behavioral regression tests: python -m pytest tests/test_base.py -q


In [15]:
# 3.4 — Smoke answers. Golden references are not passed to either generator.
if retrieval_ready and llm_ready:
    smoke_question = golden_df.iloc[0].question
    smoke_answers = support.cached_json('smoke', {
        'version': support.PROMPT_VERSION, 'question': smoke_question,
        'corpus': support.fingerprint(chunks_df), 'model': lab.GROQ_MODEL,
        'reasoning_effort': lab.GROQ_REASONING_EFFORT, 'namespace': lab.LAB_NAMESPACE},
        lambda: {'flat': lab.answer_flat_rag(smoke_question)['answer'],
                 'graph': lab.answer_graph_rag(smoke_question)['answer']})
    display(smoke_answers)
else:
    print('PENDING W4: real answers have not been generated.')

{'flat': 'The July expansion references the external provider **Cohere** – Amazon added “technology access from the startup Cohere” – and alongside it announced a new **health‑care system that can generate clinical notes after a patient visit**【chunk_id=613de2c24d2456048c08::c0000】.',
 'graph': 'The July expansion of Amazon’s AI service references the external technology provider **Cohere** (the startup whose technology is being used to build more conversational customer‑service agents) and, alongside that, Amazon announced a **health‑care system that can generate clinical notes after a patient visit**【chunk_id=613de2c24d2456048c08::c0000】.'}

# PHẦN 4 — GOLDEN DATASET & LLM-AS-A-JUDGE

Repo có 25 câu thực tế trong các CSV mang tên `golden_50`. Bản detailed chứa nguồn và reference answers.
Pipeline mặc định chọn 5 câu có nguồn đã xác minh: 1 factoid, 2 multi-hop, 2 cross-doc.
`outputs/golden_coverage.csv` phân biệt độ phủ corpus, vector và extraction.
Reference answers chỉ đưa cho Judge; không đưa vào retrieval, extraction hoặc generator.
Latency chính đo retrieval + generation; token online GraphRAG gồm seed extraction + generation.


In [16]:
# 4.1 — Verify the selected Golden; original CSV names do not imply 50 records
original_golden = pd.read_csv(lab.ROOT / 'data/graphrag_golden_50_first5000_detailed.csv')
print('Available Golden records:', len(original_golden))
display(original_golden.group.value_counts())
if data_ready:
    lab.validate_golden(golden_df)
    display(golden_df[['id', 'group', 'question']])
else:
    print('PENDING W1/W5: source coverage must be verified before choosing final Golden queries.')

Available Golden records: 25


group
multi-hop    12
cross-doc    11
factoid       2
Name: count, dtype: int64

,id,group,question
0,G5000-26,multi-hop,"What external technology provider is named inside Amazon's July AI-service expansion, and what other new AI capabili..."
1,G5000-27,cross-doc,How should the graph reconcile the statement that AMD powers multiple cloud services with the later Reuters report a...
2,G5000-28,multi-hop,"Which model providers are connected to Google Cloud Next '23 in the selected data, and which models are associated w..."
3,G5000-29,cross-doc,How did participation in White House AI commitments broaden from July to September 2023 according to the selected re...
15,G5000-41,factoid,"Which company did HPE agree to acquire to expand edge-to-cloud security, and what unified security architecture did ..."


In [17]:
# 4.2 — Judge uses the full candidate context and validates integer scores 1–5
from lab19_runtime import judge_answer, judge_json, validate_golden
print('Judge provider:', lab.JUDGE_PROVIDER, '| Model:', lab.JUDGE_MODEL)
print('Judge configured:', bool(lab.JUDGE_MODEL and lab.JUDGE_PROVIDER in lab.JUDGE_KEY_NAMES
                                and support.is_configured(lab.judge_credentials()[1])))

Judge provider: groq | Model: openai/gpt-oss-120b
Judge configured: True


In [18]:
# 4.3 — Resume by run fingerprint/query ID; cache answers before Judge calls
if retrieval_ready and llm_ready:
    eval_results_df = lab.run_evaluation(golden_df)
    display(eval_results_df)
else:
    print('PENDING W5: evaluation has not run; no scores/tokens/latency are fabricated.')

Evaluation:   0%|          | 0/5 [00:00<?, ?it/s]

Evaluation: 100%|██████████| 5/5 [00:00<?, ?it/s]

,id,group,question,reference_answer,run_id,flat_answer,graph_answer,flat_comprehensiveness,graph_comprehensiveness,flat_faithfulness,...,flat_generation_tokens,graph_generation_tokens,graph_seed_tokens,flat_retrieval_latency_s,graph_retrieval_latency_s,flat_generation_latency_s,graph_generation_latency_s,flat_judge_rationale,graph_judge_rationale,graph_supernode_events
0,G5000-26,multi-hop,"What external technology provider is named inside Amazon's July AI-service expansion, and what other new AI capabili...",Amazon's AI-service story names access to technology from Cohere. It also mentions a program for building more conve...,9ebf9bec11e988923beb337d91ab7ba120acef31b16fd694599b50d4e7d549af,The July expansion references the external provider **Cohere** – Amazon added “technology access from the startup Co...,The July expansion of Amazon’s AI service references the external technology provider **Cohere** — the startup whose...,5,5,5,...,813,677,316,0.066660,1.291701,6.783573,0.968049,The candidate correctly identifies Cohere as the external technology provider and the healthcare system that generat...,The candidate correctly identifies Cohere as the external technology provider and accurately mentions the new health...,0
1,G5000-27,cross-doc,How should the graph reconcile the statement that AMD powers multiple cloud services with the later Reuters report a...,The June 1 investment article broadly says AMD powers multiple cloud services through its chips. The June 14 Reuters...,9ebf9bec11e988923beb337d91ab7ba120acef31b16fd694599b50d4e7d549af,The graph can treat the two pieces of information as complementary rather than contradictory:\n\n* **General claim:*...,"The graph’s “AMD powers multiple cloud services” claim (from the June 1, 2023 source) describes the broader market –...",5,5,5,...,1023,821,279,0.335124,1.375014,0.982089,1.141286,The answer restates the broad claim that AMD powers multiple cloud services and correctly notes the Reuters report t...,The answer restates the broad June 1 claim that AMD powers multiple cloud services and correctly contrasts it with t...,0
2,G5000-28,multi-hop,"Which model providers are connected to Google Cloud Next '23 in the selected data, and which models are associated w...",Meta is connected via Llama 2 and Code Llama; the Technology Innovation Institute is connected via Falcon LLM; Anthr...,9ebf9bec11e988923beb337d91ab7ba120acef31b16fd694599b50d4e7d549af,- **Meta** – Llama 2 and Code Llama [chunk_id=7ca725290aab95bd56a7] \n- **Technology Innovative Institute** – Falco...,- **Meta** – provides **Llama 2** and **Code Llama**【chunk_id=7ca725290aab95bd56a7::c0000】 \n- **Technology Innovat...,5,5,5,...,897,872,437,0.546395,2.562226,1.116739,1.214025,"The candidate lists all three model providers mentioned in the reference (Meta, Technology Innovation Institute, Ant...","The candidate correctly identifies all three model providers (Meta, Technology Innovation Institute, Anthropic) ment...",0
3,G5000-29,cross-doc,How did participation in White House AI commitments broaden from July to September 2023 according to the selected re...,"The July report names seven companies including Google, Meta, and OpenAI as making voluntary AI commitments. The Sep...",9ebf9bec11e988923beb337d91ab7ba120acef31b16fd694599b50d4e7d549af,"The July 2023 report notes that **seven** technology firms – among them Google, Meta and OpenAI – had voluntarily si...","In July 2023 the White House AI‑safety pledge was signed by a core group of **seven** firms – notably Google, Meta a...",5,5,5,...,908,988,314,0.030217,6.067750,0.802907,3.532002,"The answer accurately captures the July report's seven companies and the September report's expanded list of IBM, Ad...","The answer correctly lists the July report’s seven companies (Google, Meta, OpenAI) and the September report’s addit...",0
4,G5000-41,factoid,"Which company did HPE agree to acquire to expand edge-to-cloud security, and what unified security architecture did ...",Axis

In [19]:
# 4.4 — Export comparison by group + ALL with deltas and sample counts
if eval_results_df is not None:
    comparison_df = lab.comparison_table(eval_results_df)
    comparison_df.to_csv(lab.OUTPUTS / 'graphrag_vs_flatrag_summary.csv', index=False)
    display(comparison_df)
    from lab19_reporting import generate_reports
    generate_reports(lab, eval_results_df, comparison_df)
else:
    print('PENDING W5/W6: required benchmark CSVs await real evaluation.')

,Loại câu hỏi,Metric,Flat RAG,GraphRAG,Delta Graph - Flat,Sample count,Nhận xét phân tích
0,cross-doc,Comprehensiveness,5.000,5.000,0.000,2,Hai phương pháp gần nhau.
1,cross-doc,Faithfulness,5.000,5.000,0.000,2,Hai phương pháp gần nhau.
2,cross-doc,Multi-hop reasoning,5.000,5.000,0.000,2,Hai phương pháp gần nhau.
3,cross-doc,Latency (s),1.075,6.058,4.983,2,Flat RAG thường rẻ/nhanh hơn.
4,cross-doc,Token usage,965.500,1201.000,235.500,2,Flat RAG thường rẻ/nhanh hơn.
5,factoid,Comprehensiveness,5.000,5.000,0.000,1,Hai phương pháp gần nhau.
6,factoid,Faithfulness,5.000,5.000,0.000,1,Hai phương pháp gần nhau.
7,factoid,Multi-hop reasoning,5.000,5.000,0.000,1,Hai phương pháp gần nhau.
8,factoid,Latency (s),6.908,10.152,3.244,1,Flat RAG thường rẻ/nhanh hơn.
9,factoid,Token usage,854.000,1195.000,341.000,1,Flat RAG thường rẻ/nhanh hơn.


# PHẦN 5 — FAILURE-MODE CHECKS & SUBMISSION (105–120')

Bắt buộc chứng minh:
1. Edge provenance không thiếu.
2. Entity Resolution có audit.
3. Super-node degree > 100 chỉ expand tối đa 50 edge.
4. Có comparison table.

In [20]:
# 5.1 — Report whether the actual graph exercises the supernode branch
if graph_ready:
    supernode_status = lab.test_supernode_policy()
    support.write_json(lab.OUTPUTS / 'supernode_status.json', supernode_status)
    lab.show_resolution_audit(entity_resolution_audit_df)
else:
    print('Real-data supernode and entity audit are pending; fixtures are covered by pytest.')

{'id': '18a2f0e496fb60f2b80e8775', 'name': 'Google Cloud', 'degree': 3} fetched= 3


,type,left,right,similarity,decision,reason,canonical_left,canonical_right
39,Company,Amazon Publisher Services,Amazon Web Services,0.683461,REJECT_THRESHOLD,similarity_below_threshold,Amazon Publisher Services,Amazon Web Services
0,Company,Amazon,Amazon Web Services,0.646989,REJECT_THRESHOLD,similarity_below_threshold,Amazon,Amazon Web Services
169,Company,Synopsys,Synechron,0.618264,REJECT_THRESHOLD,similarity_below_threshold,Synopsys,Synechron
131,Company,Microsoft,Salesforce,0.598405,REJECT_THRESHOLD,similarity_below_threshold,Microsoft,Salesforce
1,Company,Amazon,Amazon Publisher Services,0.585162,REJECT_THRESHOLD,similarity_below_threshold,Amazon,Amazon Publisher Services
248,Technology,AI-Powered Personal Cloud Platform,AI-Powered Customer Service Suite,0.575279,REJECT_THRESHOLD,similarity_below_threshold,AI-Powered Personal Cloud Platform,AI-Powered Customer Service Suite
204,Company,Apple,Samsung,0.575256,REJECT_THRESHOLD,similarity_below_threshold,Apple,Samsung
87,Company,IT World Canada,Worldpay,0.573572,REJECT_THRESHOLD,similarity_below_threshold,IT World Canada,Worldpay
2,Company,Amazon,Salesforce,0.537856,REJECT_THRESHOLD,similarity_below_threshold,Amazon,Salesforce
36,Company,IAS,IHSE USA,0.520189,REJECT_THRESHOLD,similarity_below_threshold,IAS,IHSE USA


High-similarity rejected pairs:


,type,left,right,similarity,decision,reason,canonical_left,canonical_right


## 5.2 — Thuyết minh kỹ thuật: học viên tự điền

1. Coreference sai ở tình huống nào?
2. Entity threshold bao nhiêu, vì sao?
3. Candidate nào similarity cao nhưng không nên merge?
4. Top 3 super-node và degree?
5. Vì sao ưu tiên edge mới nhất có thể đúng/sai?
6. Flat RAG thắng nhóm nào?
7. GraphRAG thắng nhóm nào?
8. Latency/token trade-off?
9. AI Coding Agent đề xuất gì mà bạn **không dùng**, vì sao?
10. Scale 350MB: bottleneck đầu tiên là gì?

# 🎁 BONUS

## A — Low-level / High-level
Tạo local entities và high-level topics/community reports; query router chọn tầng retrieval.

## B — Global Search via Community Reports
Nếu Neo4j instance không có GDS phù hợp, fallback:
1. export edges,
2. NetworkX community detection,
3. `UNWIND` write `community_id`,
4. LLM summarize community,
5. query global trên reports.

## C — Self-Correction Graph Retrieval
- hop 2 → LLM kiểm tra context đủ chưa,
- thiếu → hop 3,
- vẫn thiếu → vector fallback,
- bắt buộc stop condition.

In [21]:
# Bonus — Community ID alone does not meet the global-search bonus requirements
from lab19_runtime import build_communities
print('Optional scaffold only. Complete community reports/querying and quantify before claiming bonus.')

Optional scaffold only. Complete community reports/querying and quantify before claiming bonus.


In [22]:
# Bonus — Bounded hop2 → hop3 → vector fallback; enable after the core benchmark
from lab19_runtime import self_correcting_context, context_sufficient
print('Optional scaffold only. Compare quality/latency/tokens before claiming bonus.')

Optional scaffold only. Compare quality/latency/tokens before claiming bonus.


# Rubric áp dụng và trạng thái bài nộp

Theo `RUBRIC.md`: Pipeline 40đ, Failure modes 20đ, Evaluation 20đ, Technical defense/Reflection 20đ.

Chỉ đánh dấu hoàn thành sau khi chạy thật. Xem `WORKFLOW.md` và `reports/execution_status.md` để theo dõi việc đã kiểm tra và những bước còn chờ dữ liệu/secrets.
